# Notebook 5: Validating the Watchlist with Cleanlab

The resemblance watchlist (NB3) is a rank cut on the model's out-of-fold probabilities, so a natural question is whether its members are an artefact of where that cut was placed. This notebook cross-checks them with Cleanlab, which applies confident learning to the same out-of-fold probabilities but through a different decision rule: not a top-N rank cut, but a label-consistency criterion that flags given labels as statistically inconsistent with the model's confident predictions. 

Because it shares the probability basis, this is a consistency check on the selection rule rather than fully independent evidence. It confirms the watchlist is not merely an artefact of where the cutoff was placed, but it does not bring in new data. The genuinely independent cross-check follows in NB6, where the ranking is re-derived from the continuous milieu_anteil rather than the binary label. This Notebook checks how far Cleanlab's agreement reaches and explores some features of the cleanlab validation. 

In [1]:
#load libraries
import numpy as np
import pandas as pd
from cleanlab.filter import find_label_issues

/Users/lauragies/Documents/Job/NF_Bootcamp/capstone-kiezkeeper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#XGBoost OOF probabilities (527 PLR) — the basis 
oof = pd.read_csv("../../models/M_oof_xgboost.csv", dtype={"plr_id": str})
oof["plr_id"] = oof["plr_id"].astype(str).str.zfill(8)

#the provisional watchlist (25 PLR, pure oof_prob), handcoded to be validated by cleanlab 
watchlist = pd.read_csv("../../models/M2_watchlist.csv", dtype={"plr_id": str})
watchlist["plr_id"] = watchlist["plr_id"].astype(str).str.zfill(8)

print(f"{len(oof)} PLR total | {int(oof['y_true'].sum())} designated | "
      f"{len(watchlist)} on resemblance list")

527 PLR total | 109 designated | 25 on resemblance list


In [3]:
#Cleanlab needs given labels + out-of-sample pred_probs as an (n, n_classes) matrix
#Class 0 = undesignated, Class 1 = designated; our oof_prob is P(class 1)
labels = oof["y_true"].to_numpy().astype(int)
pred_probs = np.column_stack([1 - oof["oof_prob"], oof["oof_prob"]])  

#idea of cleanlab is to find_label_issues: PLR whose given label is inconsistent with the model's confident prediction
#ranked by likelihood of being an issue (most suspicious first)
issue_idx = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence",
)

#assemble the flagged PLR with their info
cl_issues = oof.iloc[issue_idx].copy()
cl_issues["cl_rank"] = np.arange(1, len(cl_issues) + 1)

#we care about issues among undesignated PLR (label 0 flagged as looking designated)
#the confident-learning analog of our early-warning watchlist
cl_undes = cl_issues[cl_issues["y_true"] == 0].copy()

print(f"Cleanlab flagged {len(cl_issues)} label issues total")
print(f"  of which {len(cl_undes)} are undesignated PLR (early-warning analog)")
print(f"  and {len(cl_issues) - len(cl_undes)} are designated PLR flagged as not-designated-looking")

Cleanlab flagged 64 label issues total
  of which 38 are undesignated PLR (early-warning analog)
  and 26 are designated PLR flagged as not-designated-looking


In [4]:
#Overlap: how many of the 25 resemblance PLR does Cleanlab independently flag?
rl_ids = set(watchlist["plr_id"])
cl_undes_ids = set(cl_undes["plr_id"])

overlap        = rl_ids & cl_undes_ids
only25         = rl_ids - cl_undes_ids
only_cleanlab  = cl_undes_ids - rl_ids

print(f"Watchlist:            {len(rl_ids)} PLR")
print(f"Cleanlab undesignated flags: {len(cl_undes_ids)} PLR")
print(f"Overlap (both agree):        {len(overlap)} PLR "
      f"({len(overlap)/len(rl_ids):.0%} of the resemblance list)")
print(f"Only on watchlist:    {len(only25)}")
print(f"Only flagged by Cleanlab:    {len(only_cleanlab)}")

if only25:
    print("\nOn resemblance list but not flagged by Cleanlab:")
    print(watchlist[watchlist["plr_id"].isin(only25)]
          [["plr_name", "bez", "oof_prob"]].to_string(index=False))

Watchlist:            25 PLR
Cleanlab undesignated flags: 38 PLR
Overlap (both agree):        25 PLR (100% of the resemblance list)
Only on watchlist:    0
Only flagged by Cleanlab:    13


In [5]:
# Cleanlabs implicit threshod for the designated class:
t1 = pred_probs[labels == 1, 1].mean()   
print(f"self-confidence threshold for class 1: {t1:.3f}")
print(f"lowest oof_prob among the 38 flagged undesignated: {cl_undes['oof_prob'].min():.3f}")

self-confidence threshold for class 1: 0.633
lowest oof_prob among the 38 flagged undesignated: 0.586


## Result 

Cleanlab flags 64 label issues in total: 26 designated areas whose labels it finds inconsistent, and 38 undesignated areas that look designation-like: the confident-learning analog of the watchlist. All 25 areas on the resemblance watchlist fall within those 38, a 100% overlap with none missed, so a formal label-consistency method independently reproduces the entire hand-built list. 

The 13 additional flags are undesignated areas Cleanlab identifies beyond our provisional cutoff of 25, which the next cells examine. Because Cleanlab reaches this agreement through a different decision rule than the rank cut, the match confirms the watchlist is not an artefact of where the cutoff was placed. Though, sharing the same probability basis, it remains a consistency check rather than fully independent evidence.

In [6]:
#where do the 13 Cleanlab-only PLR rank on oof_prob? Are they near misses? 
cl_only = oof[oof["plr_id"].isin(only_cleanlab)].copy()
cl_only["oof_rank"] = oof["oof_prob"].rank(ascending=False, method="min").astype(int)
cl_only = oof.assign(oof_rank=oof["oof_prob"].rank(ascending=False, method="min").astype(int))
print(cl_only[cl_only["plr_id"].isin(only_cleanlab)]
      .sort_values("oof_prob", ascending=False)[["plr_id","oof_prob","oof_rank"]].to_string(index=False))

  plr_id  oof_prob  oof_rank
05100316  0.766517        88
01100205  0.756666        90
04501042  0.712941        91
12100103  0.712730        92
09200716  0.704122        93
12100204  0.694774        95
01100415  0.693884        96
09100304  0.689616        98
08100101  0.658168        99
04501151  0.638546       100
04501149  0.616660       101
01200624  0.604541       103
04501043  0.586272       104


In [7]:
#to what extent are these PLRs protected? 
milieu = (pd.read_csv("../../data/Module 2/target_milieuschutz.csv", dtype={"plr_id": str})
            .assign(plr_id=lambda d: d["plr_id"].astype(str).str.zfill(8)))

cl13 = oof[oof["plr_id"].isin(only_cleanlab)].merge(
    milieu[["plr_id","milieu_anteil"]], on="plr_id", how="left")
print(cl13[["plr_id","oof_prob","milieu_anteil"]]
      .sort_values("oof_prob", ascending=False).to_string(index=False))

  plr_id  oof_prob  milieu_anteil
05100316  0.766517   1.801573e-02
01100205  0.756666   2.602949e-08
04501042  0.712941   5.816938e-09
12100103  0.712730   0.000000e+00
09200716  0.704122   0.000000e+00
12100204  0.694774   1.523684e-01
01100415  0.693884   0.000000e+00
09100304  0.689616   0.000000e+00
08100101  0.658168   1.332604e-01
04501151  0.638546   0.000000e+00
04501149  0.616660   0.000000e+00
01200624  0.604541   3.745172e-02
04501043  0.586272   5.227920e-04


In [8]:
#are the additional PLRs in cleanlab from the weak fold (Charlottenburg-Wilmersdorf, Lichtenberg)?
meta = (pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
          .assign(plr_id=lambda d: d["plr_id"].astype(str).str.zfill(8))[["plr_id","bez"]])
cl13 = cl13.merge(meta, on="plr_id", how="left")
cl13["bez_name"] = cl13["bez"].str.split(" - ").str[-1].str.strip()
cl13["weak_fold"] = cl13["bez_name"].isin(["Charlottenburg-Wilmersdorf","Lichtenberg"])
print(cl13[["plr_id","oof_prob","milieu_anteil","bez_name","weak_fold"]]
      .sort_values("oof_prob", ascending=False).to_string(index=False))
print(f"\n{cl13['weak_fold'].sum()} of {len(cl13)} Cleanlab-only PLR are in the weak fold")

  plr_id  oof_prob  milieu_anteil                   bez_name  weak_fold
05100316  0.766517   1.801573e-02                    Spandau      False
01100205  0.756666   2.602949e-08                      Mitte      False
04501042  0.712941   5.816938e-09 Charlottenburg-Wilmersdorf       True
12100103  0.712730   0.000000e+00              Reinickendorf      False
09200716  0.704122   0.000000e+00           Treptow-Köpenick      False
12100204  0.694774   1.523684e-01              Reinickendorf      False
01100415  0.693884   0.000000e+00                      Mitte      False
09100304  0.689616   0.000000e+00           Treptow-Köpenick      False
08100101  0.658168   1.332604e-01                   Neukölln      False
04501151  0.638546   0.000000e+00 Charlottenburg-Wilmersdorf       True
04501149  0.616660   0.000000e+00 Charlottenburg-Wilmersdorf       True
01200624  0.604541   3.745172e-02                      Mitte      False
04501043  0.586272   5.227920e-04 Charlottenburg-Wilmersdorf    

### Result Discussion

The 13 areas Cleanlab flags beyond the watchlist are not a separate group but the continuation of the same ranking: across all 38 undesignated flags, the oof_prob rank runs unbroken from 3 down to 104, with the 13 extras simply occupying the lower stretch (ranks 88–104, oof_prob 0.59–0.77). 

This rules out the interpretation that Cleanlab uses a fundamentally different criterion to surface a distinct band: it extends the same probability ordering past the provisional N=25 cut rather than reaching elsewhere. All 13 are essentially unprotected (milieu_anteil at or near zero, only two above 0.13: Schäfersee, 0.152 and Hasenheide, 0.133), so they are the same type of undesignated look-alike as the watchlist itself, just lower-scored. 

Of the 13, four sit in the weak cross-validation fold (Charlottenburg-Wilmersdorf): this depicts an over-representation worth noting, but not a majority, so the extras are not primarily a weak-fold artefact. 

The honest reading is therefore that Cleanlab confirms the full resemblance list (25/25) and would extend it continuously downward; the choice of where to stop remains a deliberate cutoff, not a boundary the data draws for us.

### Decision: Final watchlist with 25 Planning Areas 

The list length is a deliberate cut on a continuous ranking, not a boundary the data draws on its own. The question is which cut is best justified. We chose 25 for two reasons. First, it sits at the largest visible gap in the ranking: after rank 25 the out-of-fold probability drops from 0.824 to 0.767, the clearest step in the high-scoring range. Cleanlab's alternative of 38 falls in a stretch where scores decline smoothly, so it is anchored to Cleanlab's internal threshold rather than to any structure in our own ranking.

Second, 25 is where our two validations agree most strongly. Cleanlab confirms all 25 (100%), and the regression cross-check recovers 20 of them; at length 38 the regression overlap falls from 80% to 74%. The 25 also carry uniformly high scores (oof_prob ≥ 0.82), whereas the 13 areas beyond them sit at 0.59–0.77, with a concentration in the weak cross-validation fold (Charlottenburg-Wilmersdorf), where the model's probabilities are least reliable.

The trade-off is honest: a longer list would have been defensible, since Cleanlab flags designation-like signal continuing past rank 25, and the 25 are a strict subset of those 38. But we prioritised a tighter, higher-confidence shortlist over broader coverage for an early-warning tool meant to prompt closer examination of specific areas, precision at the top of the ranking matters more than reach.

### Further Exploration: Feature Profiles 

This next section characterises what the watchlist captures, and (as a check on the length decision) also isolates the 13 areas that a longer, Cleanlab-anchored cut (38) would have added, to see whether they share the same profile or fall off. Note this is descriptive, not causal: the areas were selected by oof_prob built from these same features, so separation between groups is expected rather than a finding. 

The watchlist-25 differ sharply from the other undesignated areas on the classic gentrification markers: Altbau share (0.50 vs 0.13), gastronomy, young-adult share and land value are all far higher. In addition, they differ only marginally from the already-designated areas on those same markers, which is exactly what a resemblance-based watchlist should do. The 13 areas beyond rank 25 share this profile on three of the four strongest markers (gastronomy, young-adult share, land value — the last in fact highest of all groups), falling off only on Altbau share (0.33 vs ~0.50). They are weaker but not qualitatively different candidates, which is why the choice between 25 and 38 was a genuine trade-off rather than a clear cut.


In [ ]:
#load features and cleanlab_38 list to check for some similarities in the lists 
meta = pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
meta["plr_id"] = meta["plr_id"].astype(str).str.zfill(8)

cleanlab_38 = (
    oof[oof["y_true"] == 0]
    .sort_values("oof_prob", ascending=False)
    .head(38)[["plr_id", "oof_prob"]]
)

In [10]:
#feature profile of the watchlist vs. (a) other undesignated PLR, (b) designated
# RAW dataset_0 values (interpretable). Descriptive only — the watchlist was selected
# by oof_prob built from these features, so differences are expected, not causal.

feats = pd.read_csv("../../data/final_datasets/dataset_0.csv", dtype={"plr_id": str})
feats["plr_id"] = feats["plr_id"].astype(str).str.zfill(8)

# attach designation label + watchlist membership
feats = feats.merge(oof[["plr_id", "y_true"]], on="plr_id", how="left")
wl_ids = set(watchlist["plr_id"])
feats["group"] = np.where(
    feats["y_true"] == 1, "designated",
    np.where(feats["plr_id"].isin(wl_ids), "watchlist", "undes_rest")
)

# feature columns only: the three axes, by prefix. Excludes id/name/bez/target/oof.
feat_cols = [c for c in feats.columns if c.startswith(("soc_", "re_", "com_"))]
print(f"{len(feat_cols)} features | groups: {feats['group'].value_counts().to_dict()}\n")

# group means (raw, interpretable)
means = feats.groupby("group")[feat_cols].mean().T
means = means[["watchlist", "undes_rest", "designated"]]   

# standardized differences (comparable across features): (meanA - meanB) / pooled SD
def std_diff(a, b):
    ma, mb = feats.loc[feats.group==a, feat_cols].mean(), feats.loc[feats.group==b, feat_cols].mean()
    sa, sb = feats.loc[feats.group==a, feat_cols].std(),  feats.loc[feats.group==b, feat_cols].std()
    pooled = np.sqrt((sa**2 + sb**2) / 2).replace(0, np.nan)
    return (ma - mb) / pooled

means["d_vs_rest"]      = std_diff("watchlist", "undes_rest")
means["d_vs_designated"] = std_diff("watchlist", "designated")

# what most separates the watchlist from the OTHER undesignated PLR
print("=== Watchlist vs. other undesignated — top separating features ===")
print(means.reindex(means["d_vs_rest"].abs().sort_values(ascending=False).index)
           [["watchlist", "undes_rest", "d_vs_rest"]].head(12).round(3).to_string())

# how the watchlist compares to the DESIGNATED areas it's meant to resemble
print("\n=== Watchlist vs. designated — where they still differ most ===")
print(means.reindex(means["d_vs_designated"].abs().sort_values(ascending=False).index)
           [["watchlist", "designated", "d_vs_designated"]].head(12).round(3).to_string())

42 features | groups: {'undes_rest': 393, 'designated': 109, 'watchlist': 25}

=== Watchlist vs. other undesignated — top separating features ===
group                                                         watchlist  undes_rest  d_vs_rest
re_altbau_share                                                   0.495       0.131      1.938
com_gastro_share_level_2026                                       0.059       0.034      1.384
soc_young_to_middle_adult_share_2025                              0.455       0.366      1.311
re_brw_trend                                                     -0.360      -0.182     -1.146
soc_average_household_size_2024                                   1.670       1.845     -1.142
re_brw_niveau                                                  2769.405    1251.822      1.017
com_n_gastro                                                     70.987      20.545      0.989
soc_transfer_benefit_share_change_2020_2024                      -0.022      -0.007     -0.917

In [11]:
#Comparison of all four groups on the strongest features above 

wl25_ids = set(watchlist["plr_id"])                    
wl38_ids = set(cleanlab_38["plr_id"])                 
extra13  = wl38_ids - wl25_ids                         

feats["group4"] = np.where(feats["y_true"]==1, "designated",
                  np.where(feats["plr_id"].isin(wl25_ids), "watchlist25",
                  np.where(feats["plr_id"].isin(extra13), "extra13", "undes_rest")))

# für die stärksten Trenner: wie sehen extra13 vs watchlist25 vs designated aus?
key = ["re_altbau_share","com_gastro_share_level_2026","soc_young_to_middle_adult_share_2025","re_brw_niveau"]
print(feats.groupby("group4")[key].mean().round(3).to_string())

             re_altbau_share  com_gastro_share_level_2026  soc_young_to_middle_adult_share_2025  re_brw_niveau
group4                                                                                                        
designated             0.517                        0.061                                 0.470       2726.895
extra13                0.333                        0.051                                 0.447       3207.498
undes_rest             0.124                        0.033                                 0.363       1184.741
watchlist25            0.495                        0.059                                 0.455       2769.405
